In [1]:
import os
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd
from PIL import Image
import torch.nn as nn
from torch import nn, optim
from torch.optim import AdamW
from torchvision import transforms
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np
import copy
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
import seaborn as sns  # 引入 seaborn 库，用于绘制更美观的统计图

# 引入模型和数据集类
from model import build_resnet as build_resnet_bn # 从 model_BatchNorm_simplified.py 导入 BatchNorm 模型构建函数，并命名为 build_resnet_bn
#from model_NoNorm import build_resnet as build_resnet_nonorm # 从 model_NoNorm.py 导入 NoNorm 模型构建函数，并命名为 build_resnet_nonorm
from dataset import MultiModalDataset

# 定义训练集复杂图片增强函数
import torchvision.transforms.functional as F # 引入 functional 模块

def train_transform_complex(image, tabular_data, label, raw_v_delta, label_min, label_max): 
    """
    应用于训练集图片的复杂增强操作：
    ... (文档字符串不变) ...
    Args:
        image (PIL.Image): 原始图片
        tabular_data (torch.Tensor): 已经标准化好的表格数据
        label (float): 原始标签
        raw_v_delta (torch.Tensor): 原始 V_delta 张量 (单个值) 
        label_min (float): 标签最小值 (用于归一化)
        label_max (float): 标签最大值 (用于归一化)
    Returns:
        list: 包含 15 个 (图片张量, 表格数据张量, 标签张量, 原始 V_delta 张量) 元组的列表 
    """
    augmented_samples = []
    # 标签归一化 
    label_normalized = (label - label_min) / (label_max - label_min)
    label_tensor = torch.tensor([label_normalized], dtype=torch.float32)

    # 转换为灰度图 
    image = image.convert("L")

    # 定义后续通用的图片处理流程 (ToTensor 和 Normalize) 
    post_crop_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ])

    # 1. 随机裁剪 5 个区域 
    for _ in range(5):
        i, j, h, w = transforms.RandomCrop.get_params(image, output_size=(336, 336))
        cropped_image = F.crop(image, i, j, h, w)

        # 2. 对裁剪后的区域进行 180° 旋转操作 
        img_rotated = F.rotate(cropped_image, 180)

        # 3. 对裁剪后的区域进行镜像操作 (新增水平镜像)
        img_mirrored = F.hflip(cropped_image)  # *** 新增：水平镜像操作 ***

        # 4. 转换为 Tensor 并归一化，并组合样本
        # 添加原始裁剪图片
        augmented_samples.append((post_crop_transform(cropped_image), tabular_data, label_tensor, raw_v_delta))
        # 添加 180 度旋转后的裁剪图片
        augmented_samples.append((post_crop_transform(img_rotated), tabular_data, label_tensor, raw_v_delta))
        # 添加 镜像后的裁剪图片
        augmented_samples.append((post_crop_transform(img_mirrored), tabular_data, label_tensor, raw_v_delta))

    return augmented_samples 

def val_test_transform_simple(image, tabular_data, label, raw_v_delta, label_min, label_max): 
    """
    应用于验证集和测试集图片的简单预处理操作：
    ... (文档字符串不变) ...
    """
    # 标签归一化 
    label_normalized = (label - label_min) / (label_max - label_min)
    label_tensor = torch.tensor([label_normalized], dtype=torch.float32)

    # 转换为灰度图并 Resize 
    image = image.convert("L")
    image = transforms.Resize((336, 336))(image)

    # 转换为 Tensor 并归一化 
    image_tensor = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ])(image)

    return [(image_tensor, tabular_data, label_tensor, raw_v_delta)] 

def custom_collate_fn(batch):
    """
    处理 from __getitem__ 返回列表的情况，先展平 batch
    """
    flattened_batch = [item for sublist in batch for item in sublist]

    images, tabular_data, labels, raw_v_deltas = zip(*flattened_batch)
    images = torch.stack(images)
    tabular_data = torch.stack(tabular_data)
    labels = torch.stack(labels)
    raw_v_deltas = torch.stack(raw_v_deltas) 

    return images, tabular_data, labels, raw_v_deltas

def train_val_test_data_process(image_dir, csv_file, v_delta_col_name='V_delta', batch_size=4, random_seed=42): 
    # 1. 读取 CSV 数据到 DataFrame
    df = pd.read_csv(csv_file)

    # 获取 V_delta 列的索引
    if v_delta_col_name in df.columns:
        v_delta_column_index = df.columns.get_loc(v_delta_col_name)
        print(f"Found V_delta column at index: {v_delta_column_index}")
    else:
        v_delta_column_index = None
        print(f"Warning: V_delta column '{v_delta_col_name}' not found in CSV. Physics regularization will not include V_delta.")

    # 计算整体的标签 min/max 值 (用于后续反归一化)
    label_min = df.iloc[:, -1].min()
    label_max = df.iloc[:, -1].max()

    # 2. 在划分数据集 *之前*， 对 DataFrame 进行 *完全随机打乱* (Shuffle)
    df_shuffled = df.sample(frac=1, random_state=random_seed).reset_index(drop=True)

    # 3. 划分训练集、验证集和测试集的 DataFrame 切片
    train_size = int(0.7 * len(df_shuffled))
    val_size = int(0.15 * len(df_shuffled))
    test_size = len(df_shuffled) - train_size - val_size

    train_df = df_shuffled.iloc[:train_size].reset_index(drop=True)
    val_df = df_shuffled.iloc[train_size:train_size + val_size].reset_index(drop=True)
    test_df = df_shuffled.iloc[train_size + val_size:].reset_index(drop=True)

    # ==================== 新增代码：将划分后的数据集保存为独立的CSV文件 ====================
    train_csv_path = 'split_train_data.csv'
    val_csv_path = 'split_val_data.csv'
    test_csv_path = 'split_test_data.csv'
    
    train_df.to_csv(train_csv_path, index=False)
    val_df.to_csv(val_csv_path, index=False)
    test_df.to_csv(test_csv_path, index=False)
    
    print(f"\n--- 数据集划分及保存状态 ---")
    print(f"✅ 训练集 ({len(train_df)} 样本) 已保存至: {train_csv_path}")
    print(f"✅ 验证集 ({len(val_df)} 样本) 已保存至: {val_csv_path}")
    print(f"✅ 测试集 ({len(test_df)} 样本) 已保存至: {test_csv_path}")
    print(f"----------------------------\n")
    # =======================================================================================

    # 新增代码：获取并打印训练集标签的最大值
    train_label_max_value = train_df.iloc[:, -1].max()
    print(f"训练集标签最大值: {train_label_max_value:.4f}")  

    # 4. 创建 *独立的* MultiModalDataset 实例
    train_dataset = MultiModalDataset(image_dir=image_dir, dataframe_slice=train_df,
                                      v_delta_column_index=v_delta_column_index, 
                                      transform=train_transform_complex,
                                      label_min=label_min, label_max=label_max)

    # 获取训练集计算出的 scaler (用于验证集和测试集的 transform)
    scaler = train_dataset.scaler

    val_dataset = MultiModalDataset(image_dir=image_dir, dataframe_slice=val_df,
                                    v_delta_column_index=v_delta_column_index, 
                                    transform=val_test_transform_simple, scaler=scaler,
                                    label_min=label_min, label_max=label_max)

    test_dataset = MultiModalDataset(image_dir=image_dir, dataframe_slice=test_df,
                                     v_delta_column_index=v_delta_column_index, 
                                     transform=val_test_transform_simple, scaler=scaler,
                                     label_min=label_min, label_max=label_max)

    # 5. 创建 DataLoader
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=custom_collate_fn)
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=custom_collate_fn)
    test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=custom_collate_fn)

    return train_dataloader, val_dataloader, test_dataloader, label_min, label_max, v_delta_column_index 


def train_model_process(model, train_dataloader, val_dataloader, test_dataloader,
                        label_min, label_max, v_delta_column_index, 
                        L_max, alpha, v_delta_opt, lambda_reg, 
                        num_epoch, avg_true):

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    criterion_task = nn.SmoothL1Loss()
    criterion_phys = nn.SmoothL1Loss() 

    model = model.to(device)

    best_model_wts = copy.deepcopy(model.state_dict())
    best_loss = float('inf') 

    train_loss_all = [] 
    train_task_loss_all = [] 
    train_phys_loss_all = [] 
    val_loss_all = [] 
    val_task_loss_all = [] 
    val_phys_loss_all = [] 

    train_mae_all = []
    val_mae_all = []
    train_r2_all = []
    val_r2_all = []
    train_outputs_all_epochs = []
    train_labels_all_epochs = []
    val_outputs_all_epochs = []
    val_labels_all_epochs = []

    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.3, patience=3, verbose=True)


    for epoch in range(num_epoch):
        print(f'Epoch {epoch + 1}/{num_epoch}')
        print('-' * 10)
        print(f"真实值平均值: {avg_true:.4f}")

        # 训练阶段
        model.train()
        epoch_train_total_loss = 0.0
        epoch_train_task_loss = 0.0
        epoch_train_phys_loss = 0.0
        epoch_train_mae = 0.0
        train_samples = 0
        all_train_outputs = []
        all_train_labels = []

        for images, tabular_data, labels, raw_v_deltas in train_dataloader:
            images = images.to(device).float()
            tabular_data = tabular_data.to(device).float()
            labels = labels.to(device).float().squeeze(-1)
            raw_v_deltas = raw_v_deltas.to(device).float() 

            optimizer.zero_grad()
            L_model_normalized, L_phys_normalized, _, _, _ = model(
                images, tabular_data, raw_v_deltas,
                L_max=L_max, alpha=alpha, v_delta_opt=v_delta_opt,
                label_min=label_min, label_max=label_max
            ) 

            labels_normalized = labels.unsqueeze(-1) 

            task_loss = criterion_task(L_model_normalized, labels_normalized)
            phys_loss = criterion_phys(L_model_normalized, L_phys_normalized)
            total_loss = task_loss + lambda_reg * phys_loss 

            total_loss.backward()
            optimizer.step()

            epoch_train_total_loss += total_loss.item() * labels.size(0) 
            epoch_train_task_loss += task_loss.item() * labels.size(0)
            epoch_train_phys_loss += phys_loss.item() * labels.size(0)

            outputs_original = L_model_normalized.squeeze(-1) * (label_max - label_min) + label_min 
            labels_original = labels * (label_max - label_min) + label_min  

            epoch_train_mae += mean_absolute_error(labels_original.cpu().numpy(), outputs_original.cpu().detach().numpy()) * labels.size(0) 
            all_train_outputs.append(outputs_original.cpu().detach().numpy())
            all_train_labels.append(labels_original.cpu().numpy())
            train_samples += labels.size(0) 

        # 计算 epoch 平均损失和指标
        train_epoch_outputs = np.concatenate(all_train_outputs, axis=0)
        train_epoch_labels = np.concatenate(all_train_labels, axis=0)
        train_r2 = r2_score(train_epoch_labels, train_epoch_outputs)
        train_total_loss = epoch_train_total_loss / train_samples if train_samples > 0 else 0
        train_task_loss = epoch_train_task_loss / train_samples if train_samples > 0 else 0
        train_phys_loss = epoch_train_phys_loss / train_samples if train_samples > 0 else 0
        train_mae = epoch_train_mae / train_samples if train_samples > 0 else 0

        train_loss_all.append(train_total_loss)
        train_task_loss_all.append(train_task_loss)
        train_phys_loss_all.append(train_phys_loss)
        train_mae_all.append(train_mae)
        train_r2_all.append(train_r2)
        train_outputs_all_epochs.append(train_epoch_outputs)
        train_labels_all_epochs.append(train_epoch_labels)

        # 验证阶段
        model.eval()
        epoch_val_total_loss = 0.0
        epoch_val_task_loss = 0.0
        epoch_val_phys_loss = 0.0
        epoch_val_mae = 0.0
        val_samples = 0
        all_val_outputs = []
        all_val_labels = []

        with torch.no_grad():
            for images, tabular_data, labels, raw_v_deltas in val_dataloader:
                images = images.to(device).float()
                tabular_data = tabular_data.to(device).float()
                labels = labels.to(device).float().squeeze(-1)
                raw_v_deltas = raw_v_deltas.to(device).float()

                L_model_normalized, L_phys_normalized, _, _, _ = model(
                     images, tabular_data, raw_v_deltas,
                     L_max=L_max, alpha=alpha, v_delta_opt=v_delta_opt,
                     label_min=label_min, label_max=label_max
                ) 

                labels_normalized = labels.unsqueeze(-1) 

                task_loss = criterion_task(L_model_normalized, labels_normalized)
                phys_loss = criterion_phys(L_model_normalized, L_phys_normalized)
                total_loss = task_loss + lambda_reg * phys_loss

                epoch_val_total_loss += total_loss.item() * labels.size(0)
                epoch_val_task_loss += task_loss.item() * labels.size(0)
                epoch_val_phys_loss += phys_loss.item() * labels.size(0)

                outputs_original = L_model_normalized.squeeze(-1) * (label_max - label_min) + label_min
                labels_original = labels * (label_max - label_min) + label_min

                epoch_val_mae += mean_absolute_error(labels_original.cpu().numpy(), outputs_original.cpu().detach().numpy()) * labels.size(0)
                all_val_outputs.append(outputs_original.cpu().detach().numpy())
                all_val_labels.append(labels_original.cpu().numpy())
                val_samples += labels.size(0)

        # 计算 epoch 平均损失和指标
        val_epoch_outputs = np.concatenate(all_val_outputs, axis=0)
        val_epoch_labels = np.concatenate(all_val_labels, axis=0)
        val_r2 = r2_score(val_epoch_labels, val_epoch_outputs)
        val_total_loss = epoch_val_total_loss / val_samples if val_samples > 0 else 0
        val_task_loss = epoch_val_task_loss / val_samples if val_samples > 0 else 0
        val_phys_loss = epoch_val_phys_loss / val_samples if val_samples > 0 else 0

        if val_samples > 0:
            val_mae = epoch_val_mae / val_samples
        else:
            val_mae = np.nan

        val_mae = np.nan_to_num(val_mae, nan=0.0)

        val_loss_all.append(val_total_loss)
        val_task_loss_all.append(val_task_loss)
        val_phys_loss_all.append(val_phys_loss)
        val_mae_all.append(val_mae)
        val_r2_all.append(val_r2)
        val_outputs_all_epochs.append(val_epoch_outputs)
        val_labels_all_epochs.append(val_epoch_labels)

        train_relative_error = (train_mae / avg_true) * 40
        val_relative_error = (val_mae / avg_true) * 40

        print(f'Train Total Loss: {train_total_loss:.4f}, Task Loss: {train_task_loss:.4f}, Phys Loss: {train_phys_loss:.4f}, MAE: {train_mae:.4f} ({train_relative_error:.2f}%), R²: {train_r2:.4f}')
        print(f'Val Total Loss: {val_total_loss:.4f}, Task Loss: {val_task_loss:.4f}, Phys Loss: {val_phys_loss:.4f}, MAE: {val_mae:.4f} ({val_relative_error:.2f}%), R²: {val_r2:.4f}')

        scheduler.step(val_total_loss)

        if val_total_loss < best_loss:
            best_loss = val_total_loss
            best_model_wts = copy.deepcopy(model.state_dict())

    # 加载最佳模型权重 
    model.load_state_dict(best_model_wts)

    # 测试阶段 
    model.eval()
    epoch_test_total_loss = 0.0  
    test_samples = 0
    all_test_outputs = []
    all_test_labels = []

    with torch.no_grad():
        for images, tabular_data, labels, raw_v_deltas in test_dataloader:
            images = images.to(device).float()
            tabular_data = tabular_data.to(device).float()
            labels = labels.to(device).float().squeeze(-1)
            raw_v_deltas = raw_v_deltas.to(device).float()

            L_model_normalized, L_phys_normalized, _, _, _ = model(
                 images, tabular_data, raw_v_deltas,
                 L_max=L_max, alpha=alpha, v_delta_opt=v_delta_opt,
                 label_min=label_min, label_max=label_max
            )

            labels_normalized = labels.unsqueeze(-1) 

            task_loss = criterion_task(L_model_normalized, labels_normalized)
            phys_loss = criterion_phys(L_model_normalized, L_phys_normalized)
            total_loss = task_loss + lambda_reg * phys_loss
            epoch_test_total_loss += total_loss.item() * labels.size(0)

            outputs_original = L_model_normalized.squeeze(-1) * (label_max - label_min) + label_min
            labels_original = labels * (label_max - label_min) + label_min

            all_test_outputs.append(outputs_original.cpu().detach().numpy())
            all_test_labels.append(labels_original.cpu().numpy())
            test_samples += labels.size(0)

    test_outputs = np.concatenate(all_test_outputs, axis=0)
    test_labels = np.concatenate(all_test_labels, axis=0)
    test_r2 = r2_score(test_labels, test_outputs)
    test_mae = mean_absolute_error(test_labels, test_outputs)
    test_total_loss = epoch_test_total_loss / test_samples if test_samples > 0 else 0


    return {
        'train_loss': train_loss_all, 
        'train_task_loss': train_task_loss_all, 
        'train_phys_loss': train_phys_loss_all, 
        'val_loss': val_loss_all, 
        'val_task_loss': val_task_loss_all, 
        'val_phys_loss': val_phys_loss_all, 
        'train_mae': train_mae_all,
        'val_mae': val_mae_all,
        'train_r2': train_r2_all,
        'val_r2': val_r2_all,
        'test_loss': test_total_loss, 
        'test_mae': test_mae,
        'test_r2': test_r2,
        'train_outputs_all_epochs': train_outputs_all_epochs,
        'train_labels_all_epochs': train_labels_all_epochs,
        'val_outputs_all_epochs': val_outputs_all_epochs,
        'val_labels_all_epochs': val_labels_all_epochs,
        'test_outputs': test_outputs,
        'test_labels': test_labels
    }

def matplo_loss(results, model_name):
    plt.figure(figsize=(10, 6))
    plt.plot(results['train_loss'], label=f'{model_name} Train Loss')
    plt.plot(results['val_loss'], label=f'{model_name} Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(f'{model_name} Training and Validation Loss')
    plt.legend()
    plt.grid(True)

def plot_r2_scatter(true_labels, predictions, title, r2, filename):
    plt.figure(figsize=(8, 8))
    sns.scatterplot(x=true_labels, y=predictions)
    plt.xlabel('True Values')
    plt.ylabel('Predictions')
    plt.title(f'{title}\nR² = {r2:.4f}')
    lims = [
        np.min([plt.xlim(), plt.ylim()]),  
        np.max([plt.xlim(), plt.ylim()]),  
    ]
    plt.plot(lims, lims, 'k-', alpha=0.75, zorder=0)
    plt.xlim(lims)
    plt.ylim(lims)
    plt.grid(True)
    plt.savefig(filename) # 保存散点图

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    image_dir = './GHdata'
    csv_file = './data.csv'
    batch_size = 12
    num_epoch = 20

    # 物理公式参数和正则化权重
    L_max_physics = 19 
    alpha_physics = 0.001 
    v_delta_opt_physics = 0.05 
    lambda_regularization = 0.1

    # V_delta 对应的列名 
    v_delta_column_name = 'δcontent(after)' 

    # 获取数据加载器和标签的 min/max 值, 以及 V_delta 列索引
    train_loader, val_loader, test_loader, label_min, label_max, v_delta_col_index = train_val_test_data_process(
        image_dir, csv_file, v_delta_col_name=v_delta_column_name, batch_size=batch_size)

    # 计算真实值平均值 
    df_full = pd.read_csv(csv_file)
    avg_true = np.mean(df_full.iloc[:, -1].values.astype('float32'))
    print(f"真实值平均值: {avg_true:.4f}")

    # 1. 创建 BatchNorm 模型
    model_bn = build_resnet_bn(tabular_input_dim=30).to(device) 
    models = {'BatchNorm_PhysicsReg': model_bn} 


    model_results = {}

    # 2. 循环训练和评估不同的模型
    for model_name, model in models.items():
        print(f"\n--- Training and Evaluating {model_name} Model ---")

        train_process = train_model_process(
            model=model,
            train_dataloader=train_loader,
            val_dataloader=val_loader,
            test_dataloader=test_loader,
            label_min=label_min, 
            label_max=label_max, 
            v_delta_column_index=v_delta_col_index, 
            L_max=L_max_physics, 
            alpha=alpha_physics,
            v_delta_opt=v_delta_opt_physics,
            lambda_reg=lambda_regularization, 
            num_epoch=num_epoch,
            avg_true=avg_true
        )
        model_results[model_name] = train_process

        # 3. 绘制 Loss 曲线图 
        plt.figure(figsize=(12, 8))
        plt.plot(train_process['train_loss'], label=f'{model_name} Train Total Loss')
        plt.plot(train_process['val_loss'], label=f'{model_name} Val Total Loss')
        plt.plot(train_process['train_task_loss'], label=f'{model_name} Train Task Loss', linestyle='--')
        plt.plot(train_process['val_task_loss'], label=f'{model_name} Val Task Loss', linestyle='--')
        plt.plot(train_process['train_phys_loss'], label=f'{model_name} Train Physics Loss', linestyle=':')
        plt.plot(train_process['val_phys_loss'], label=f'{model_name} Val Physics Loss', linestyle=':')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title(f'{model_name} Training and Validation Losses (Total, Task, Physics)')
        plt.legend()
        plt.grid(True)
        plt.savefig(f'losses_curve_{model_name}.png') 
        plt.show()

        # 4. 打印测试集性能 
        test_loss = train_process['test_loss'] 
        test_mae = train_process['test_mae']
        test_r2 = train_process['test_r2']
        print(f"{model_name} Test Total Loss: {test_loss:.4f}, Test MAE: {test_mae:.4f}, Test R²: {test_r2:.4f}")

        # 5. 找到验证集损失最小的 epoch 
        best_val_loss_epoch = np.argmin(train_process['val_loss']) 
        best_val_total_loss = train_process['val_loss'][best_val_loss_epoch]
        best_val_task_loss = train_process['val_task_loss'][best_val_loss_epoch]
        best_val_phys_loss = train_process['val_phys_loss'][best_val_loss_epoch]

        print(f"{model_name} Best Epoch (based on Val Total Loss): {best_val_loss_epoch + 1}")
        print(f"  Best Val Total Loss: {best_val_total_loss:.4f}")
        print(f"  Corresponding Val Task Loss: {best_val_task_loss:.4f}")
        print(f"  Corresponding Val Physics Loss: {best_val_phys_loss:.4f}")


    # 6.  绘制 R² 散点图
    for model_name, train_process in model_results.items():
        print(f"\n--- Plotting R² Scatter Plots for {model_name} Model ---")
        best_val_epoch_idx = np.argmin(train_process['val_loss']) 

        # 1. 最佳验证 Total Loss Epoch 的 训练集 R² 散点图
        best_train_outputs = train_process['train_outputs_all_epochs'][best_val_epoch_idx]
        best_train_labels = train_process['train_labels_all_epochs'][best_val_epoch_idx]
        best_train_r2 = r2_score(best_train_labels, best_train_outputs)
        plot_r2_scatter(best_train_labels, best_train_outputs,
                        f"{model_name} Train Dataset (Epoch {best_val_epoch_idx + 1})",
                        best_train_r2,
                        f"{model_name}_train_r2_scatter_best_val_epoch.png")
        plt.show()

        # 2. 最佳验证 Total Loss Epoch 的 验证集 R² 散点图
        best_val_outputs = train_process['val_outputs_all_epochs'][best_val_epoch_idx]
        best_val_labels = train_process['val_labels_all_epochs'][best_val_epoch_idx]
        best_val_r2 = r2_score(best_val_labels, best_val_outputs)
        plot_r2_scatter(best_val_labels, best_val_outputs,
                        f"{model_name} Validation Dataset (Epoch {best_val_epoch_idx + 1})",
                        best_val_r2,
                        f"{model_name}_val_r2_scatter_best_val_epoch.png")
        plt.show()

        # 3. 最后 Epoch 的 测试集 R² 散点图
        test_outputs = train_process['test_outputs']
        test_labels = train_process['test_labels']
        test_r2 = train_process['test_r2']
        plot_r2_scatter(test_labels, test_outputs,
                        f"{model_name} Test Dataset",
                        test_r2,
                        f"{model_name}_test_r2_scatter_test_set.png")
        plt.show()


if __name__ == '__main__':
    main()

/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Found V_delta column at index: 24

--- 数据集划分及保存状态 ---
✅ 训练集 (93 样本) 已保存至: split_train_data.csv
✅ 验证集 (20 样本) 已保存至: split_val_data.csv
✅ 测试集 (21 样本) 已保存至: split_test_data.csv
----------------------------

训练集标签最大值: 70.5700
真实值平均值: 62.3752

--- Training and Evaluating BatchNorm_PhysicsReg Model ---
Epoch 1/20
----------
真实值平均值: 62.3752


/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "


RuntimeError: mat1 and mat2 shapes cannot be multiplied (180x25 and 29x128)

In [2]:
import os
import shutil
import pandas as pd

def physical_split_images(source_img_dir, target_base_dir):
    """
    根据 CSV 划分文件，物理移动/拷贝图片到独立文件夹
    """
    # 1. 定义划分文件和对应的目标文件夹名
    splits = {
        'train': 'split_train_data.csv',
        'val': 'split_val_data.csv',
        'test': 'split_test_data.csv'
    }

    # 检查源目录是否存在
    if not os.path.exists(source_img_dir):
        print(f"❌ 错误：找不到原始图片目录 {source_img_dir}")
        return

    print(f"🚀 开始物理划分图片...")

    for split_name, csv_file in splits.items():
        # 检查 CSV 是否存在
        if not os.path.exists(csv_file):
            print(f"⚠️ 警告：找不到划分文件 {csv_file}，跳过此部分。")
            continue

        # 读取 CSV
        df = pd.read_csv(csv_file)
        
        # 假设第一列是图片文件名 (例如: 'image_01.png')
        img_column = df.columns[0]
        image_list = df[img_column].tolist()

        # 创建目标文件夹 (例如: ./Split_Images/train)
        target_dir = os.path.join(target_base_dir, split_name)
        os.makedirs(target_dir, exist_ok=True)

        print(f"   -> 正在处理 {split_name} 集，共 {len(image_list)} 张图片...")

        count = 0
        for img_name in image_list:
            src_path = os.path.join(source_img_dir, img_name)
            dst_path = os.path.join(target_dir, img_name)

            if os.path.exists(src_path):
                # 使用 shutil.copy 而不是 move，保留原始 GHdata 目录不变
                shutil.copy(src_path, dst_path)
                count += 1
            else:
                print(f"      ❌ 找不到图片: {src_path}")

        print(f"   ✅ 完成！成功拷贝 {count} 张图片到 {target_dir}")

    print("\n🎉 所有图片物理划分任务已完成！")
    print(f"📂 您的新目录结构如下：")
    print(f"{target_base_dir}/")
    print(f"  ├── train/  ({len(os.listdir(os.path.join(target_base_dir, 'train'))) if os.path.exists(os.path.join(target_base_dir, 'train')) else 0} 张)")
    print(f"  ├── val/    ({len(os.listdir(os.path.join(target_base_dir, 'val'))) if os.path.exists(os.path.join(target_base_dir, 'val')) else 0} 张)")
    print(f"  └── test/   ({len(os.listdir(os.path.join(target_base_dir, 'test'))) if os.path.exists(os.path.join(target_base_dir, 'test')) else 0} 张)")

if __name__ == '__main__':
    # 配置路径
    ORIGINAL_GH_DATA = './GHdata'      # 您原始图片存放的目录
    OUTPUT_FOLDER = './Split_Images'   # 想要存放划分后图片的根目录
    
    physical_split_images(ORIGINAL_GH_DATA, OUTPUT_FOLDER)

🚀 开始物理划分图片...
   -> 正在处理 train 集，共 93 张图片...
   ✅ 完成！成功拷贝 93 张图片到 ./Split_Images/train
   -> 正在处理 val 集，共 20 张图片...
   ✅ 完成！成功拷贝 20 张图片到 ./Split_Images/val
   -> 正在处理 test 集，共 21 张图片...
   ✅ 完成！成功拷贝 21 张图片到 ./Split_Images/test

🎉 所有图片物理划分任务已完成！
📂 您的新目录结构如下：
./Split_Images/
  ├── train/  (93 张)
  ├── val/    (20 张)
  └── test/   (21 张)
